<!-- circuitkit-logo -->
<p align="center">
  <picture>
    <source media="(prefers-color-scheme: dark)" srcset="../../docs/assets/circuitkit-logo-white.png">
    <img src="../../docs/assets/circuitkit-logo-black.png" width="220" alt="CircuitKIT">
  </picture>
</p>

# 📊 Evaluation Deep Dive — The 6-Pillar Faithfulness Framework

CircuitKIT evaluates discovered circuits using 6 complementary pillars. Each answers a different question about whether the circuit truly explains the model's behavior.

| # | Pillar | Question | Cost |
|---|--------|----------|------|
| 1 | Causal Patching | Does ablating the circuit break the behavior? | Fast |
| 2 | Ablation | Is the circuit sufficient to reproduce the behavior? | Fast |
| 3 | Stability | Is the circuit reproducible across random seeds? | Expensive |
| 4 | Robustness | Is the circuit stable under different corruptions? | Moderate |
| 5 | Baselines | Does the circuit beat random/magnitude baselines? | Moderate |
| 6 | Generalization | Does the circuit transfer to related tasks? | Expensive |

- **Model:** `meta-llama/Llama-3.2-1B` (~1B params)
- **Task:** IOI (with `double_io` as generalization target)
- **Runtime:** ~25-35 min on Colab T4

---

## Setup

In [1]:
# ⚠️ Must be set before any CUDA context is created.
# If you've already run GPU cells in a prior notebook, restart the runtime first.
%env PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True

import warnings
warnings.filterwarnings('ignore', module='transformer_lens')
warnings.filterwarnings('ignore', module='lm_eval')
warnings.filterwarnings('ignore', message='.*pretrained.*model kwarg.*')

import gc
import json
import torch
from circuitkit import Pipeline

print(f"CUDA: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only'}")

env: PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
CUDA: NVIDIA A100 80GB PCIe


## Step 1: Discover a Circuit

We need a circuit before we can evaluate it.

In [2]:
pipe = Pipeline(
    "meta-llama/Llama-3.2-1B",
    task="ioi",
    output_dir="./results/eval_deep_dive",
)

pipe.discover(
    algorithm="eap-ig",
    level="node",
    sparsity=0.3,
    n_examples=256,
    batch_size=2,
    ig_steps=5,
)

print(pipe._circuit)
print(f"Artifact: {pipe._artifact_path}")

Loaded pretrained model meta-llama/Llama-3.2-1B into HookedTransformer
06:55:31 • ==================================================
06:55:31 • Starting: Circuit Discovery
06:55:31 • ==================================================
06:55:31 • [1/4] Loading and validating configuration
06:55:31 ⚙ Configuration validated successfully.
06:55:31 ⚙ Config: EAP-IG on ioi task, node level, 30% sparsity
06:55:31 • [2/4] Setting up model (model=meta-llama/Llama-3.2-1B)
Loaded pretrained model meta-llama/Llama-3.2-1B into HookedTransformer
06:55:36 🔧 Model loading: 4.94s
06:55:36 🔧 Model: meta-llama/Llama-3.2-1B (1.5B params, cuda device)
06:55:36 • [3/4] Running discovery algorithm (algorithm=eap-ig)
06:55:36 • Starting EAP-IG discovery algorithm
06:55:36 • Generating new IOI data for Llama-3.2-1B (samples=256, seed=42)
06:55:37 • Saved generated data to cache: cache/ioi/ioi_Llama-3.2-1B_256_seed42.csv


100%|██████████| 128/128 [01:04<00:00,  1.98it/s]

06:56:46 • Converted 528 node scores from EAP-IG graph to CircuitKIT format.
06:56:46 • Saved unified CircuitScores → results/eval_deep_dive/eap-ig_meta-llama_Llama-3.2-1B_ioi_node_scores.json
06:56:46 • Saved 'scores' → results/eval_deep_dive/eap-ig_meta-llama_Llama-3.2-1B_ioi_node_scores.pt
06:56:46 • [4/4] Identifying nodes to prune
06:56:46 • 
--- Running Categorized Node Pruning (Scope: BOTH) ---
06:56:46 • Found 512 Attention Heads and 16 MLP layers to consider for pruning.
06:56:46 • Pruning 4 out of 16 MLP layers (30%).
06:56:46 • Targeting to prune 153 out of 512 total Attention Heads (30%).
06:56:46 • Actually pruned 153 Attention Heads.
06:56:46 •   - Layer 0: Pruned 16/32 heads.
06:56:46 •   - Layer 1: Pruned 16/32 heads.
06:56:46 •   - Layer 2: Pruned 16/32 heads.
06:56:46 •   - Layer 3: Pruned 8/32 heads.
06:56:46 •   - Layer 4: Pruned 13/32 heads.
06:56:46 •   - Layer 5: Pruned 10/32 heads.
06:56:46 •   - Layer 6: Pruned 5/32 heads.
06:56:46 •   - Layer 7: Pruned 8/32 he

## Section A: Pillars 1 & 2 — Patching and Ablation (Fast)

These are the cheapest evaluations and should always be run first.

- **Patching**: Ablate (zero out) the circuit nodes → measure how much the target behavior degrades. A faithful circuit should show a large performance drop.
- **Ablation**: Keep only the circuit nodes, ablate everything else → measure retained performance. A sufficient circuit should retain most of the behavior.

In [3]:
pipe.evaluate(
    pillars=["patching", "ablation"],
    n_examples=256,
)

if pipe._eval_report:
    print(json.dumps(pipe._eval_report, indent=2, default=str))

06:56:46 • ==================================================
06:56:46 • Starting: Circuit Evaluation
06:56:46 • ==================================================
06:56:46 • [1/4] Loading config and model
06:56:46 ⚙ Configuration validated successfully.
Loaded pretrained model meta-llama/Llama-3.2-1B into HookedTransformer
06:56:52 🔧 Model loading: 5.39s
06:56:52 • Loading cached IOI data for Llama-3.2-1B from cache/ioi/ioi_Llama-3.2-1B_256_seed42.csv
06:56:52 • [2/4] Reconstructing circuit
06:56:56 • [3/4] Running faithfulness evaluation
06:56:56 • Starting full faithfulness evaluation with pillars: ['patching', 'ablation']
06:56:56 🔧 Model: Llama-3.2-1B
06:56:56 • Task: ioi
06:56:56 • Algorithm: eap-ig
06:56:56 • Computing Pillar 1: Causal Patching...
06:56:56 • Pillar 1: Running causal patching evaluation...


100%|██████████| 128/128 [00:09<00:00, 13.01it/s]

06:57:20 • Pillar 1 Score (Causal Patching faithfulness ratio): 0.9582 (raw metric: 1.6250, clean: 1.6953, corrupt: 0.0133)
06:57:20 •   Patching faithfulness ratio: 0.9582 (raw metric: 1.6250) (24.2s)


06:57:21 • Computing Pillar 2: Ablation...
06:57:21 • Pillar 2: Running ablation evaluation (intervention: zero)...


100%|██████████| 128/128 [00:06<00:00, 20.19it/s]

06:57:41 • Pillar 2 Score (Ablation - zero faithfulness ratio): 0.0365 (raw metric: 0.0747, clean: 1.6953, corrupt: 0.0133)
06:57:41 •   Ablation faithfulness ratio: 0.0365 (raw metric: 0.0747) (20.5s)


06:57:42 • ======================================================================
06:57:42 ✓ FAITHFULNESS EVALUATION COMPLETE
06:57:42 • ======================================================================
06:57:42 • Total time: 45.5s
06:57:42 • Pillar timings:
06:57:42 •   patching       :    24.2s
06:57:42 •   ablation       :    20.5s
06:57:42 • ======================================================================
06:57:42 ✓ Full faithfulness report complete
06:57:42 • ==================================================
06:57:42 ✓ Completed: Circuit Evaluation in 55.4s
06:57:42 • ==================================================
"============================================================\nFAITHFULNESS EVALUATION REPORT\n============================================================\n\nPILLAR 1: Causal Patching Validity\n  Score: 0.9582\n\nPILLAR 2: Ablation Faithfulness\n  Score: 0.0365\n\nMETADATA\n  algorithm: eap-ig\n  model: Llama-3.2-1B\n  task: ioi\n  level: node\n  scope: 

In [4]:
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## Section B: Pillar 5 — Baseline Comparison

Does the circuit beat random selection? Pillar 5 compares the discovered circuit's scores against random and magnitude-based baselines at the same sparsity level.

In [5]:
pipe.evaluate(
    pillars=["baselines"],
    n_examples=256,
)

if pipe._eval_report:
    print(json.dumps(pipe._eval_report, indent=2, default=str))

06:57:42 • ==================================================
06:57:42 • Starting: Circuit Evaluation
06:57:42 • ==================================================
06:57:42 • [1/4] Loading config and model
06:57:42 ⚙ Configuration validated successfully.
Loaded pretrained model meta-llama/Llama-3.2-1B into HookedTransformer
06:57:47 🔧 Model loading: 4.81s
06:57:47 • Loading cached IOI data for Llama-3.2-1B from cache/ioi/ioi_Llama-3.2-1B_256_seed42.csv
06:57:47 • [2/4] Reconstructing circuit
06:57:51 • [3/4] Running faithfulness evaluation
06:57:51 • Starting full faithfulness evaluation with pillars: ['baselines']
06:57:51 🔧 Model: Llama-3.2-1B
06:57:51 • Task: ioi
06:57:51 • Algorithm: eap-ig
06:57:51 • Computing Pillar 5: Baseline Comparison...
06:57:51 • Pillar 5: Running baselines comparison (random, magnitude)...
06:57:51 •   Circuit sparsity: 69.81%
06:57:51 • Evaluating discovered circuit...


100%|██████████| 128/128 [00:09<00:00, 13.12it/s]

06:58:01 •   Circuit score: 1.6250
06:58:01 • Evaluating random baseline...



100%|██████████| 128/128 [00:07<00:00, 17.29it/s]

06:59:07 •   Random baseline score: 0.2401
06:59:07 • Evaluating magnitude baseline...



100%|██████████| 128/128 [00:09<00:00, 13.10it/s]

06:59:21 •   Magnitude baseline score: 0.0270
06:59:21 • 
Baseline Comparison Summary:
06:59:21 •   Circuit:          1.6250
06:59:21 •   Random: 0.2401 (improvement: 6.77x, percentage: 676.8)
06:59:21 •   Magnitude: 0.0270 (improvement: 60.24x, percentage: 6023.5)
06:59:21 •   Advantage:        1.3849
06:59:21 • 
Summary: Circuit substantially outperforms random baseline (6.77x improvement)
06:59:21 ✓   Baseline comparison complete (89.8s)


06:59:22 •     Circuit substantially outperforms random baseline (6.77x improvement)
06:59:22 • ======================================================================
06:59:22 ✓ FAITHFULNESS EVALUATION COMPLETE
06:59:22 • ======================================================================
06:59:22 • Total time: 90.8s
06:59:22 • Pillar timings:
06:59:22 •   baselines      :    89.8s
06:59:22 • ======================================================================
06:59:22 ✓ Full faithfulness report complete
06:59:22 • ==================================================
06:59:22 ✓ Completed: Circuit Evaluation in 99.8s
06:59:22 • ==================================================
"============================================================\nFAITHFULNESS EVALUATION REPORT\n============================================================\n\nPILLAR 1: Causal Patching Validity\n  Score: N/A\n\nPILLAR 2: Ablation Faithfulness\n  Score: N/A\n\nPILLAR 5: Baseline Comparison\n  circuit_score: 1.6

In [6]:
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## Section C: Pillar 4 — Robustness

How stable is the circuit under different corruption strategies? Pillar 4 varies the corruption method and checks whether the same nodes remain important.

In [7]:
pipe.evaluate(
    pillars=["robustness"],
    n_examples=256,
)

if pipe._eval_report:
    print(json.dumps(pipe._eval_report, indent=2, default=str))

06:59:23 • ==================================================
06:59:23 • Starting: Circuit Evaluation
06:59:23 • ==================================================
06:59:23 • [1/4] Loading config and model
06:59:23 ⚙ Configuration validated successfully.
Loaded pretrained model meta-llama/Llama-3.2-1B into HookedTransformer
06:59:28 🔧 Model loading: 4.77s
06:59:28 • Loading cached IOI data for Llama-3.2-1B from cache/ioi/ioi_Llama-3.2-1B_256_seed42.csv
06:59:28 • Loading cached IOI data for Llama-3.2-1B from cache/ioi/ioi_Llama-3.2-1B_256_seed42.csv
06:59:28 • [2/4] Reconstructing circuit
06:59:32 • [3/4] Running faithfulness evaluation
06:59:32 • Starting full faithfulness evaluation with pillars: ['robustness']
06:59:32 🔧 Model: Llama-3.2-1B
06:59:32 • Task: ioi
06:59:32 • Algorithm: eap-ig
06:59:32 • Computing Pillar 4: Robustness to corruptions (['paraphrase', 'entity_swap', 'distractor', 'role_swap', 'token_swap', 'logical_negation', 'format_distractor', 'position_shift'])...
06:5

100%|██████████| 128/128 [00:06<00:00, 20.27it/s]

06:59:38 • Pillar 4 Original Score: 0.0747
06:59:38 • Evaluating circuit on paraphrase corrupted data...



100%|██████████| 128/128 [00:06<00:00, 20.28it/s]

06:59:45 • Pillar 4 Variant Score (paraphrase): 0.0747
06:59:45 • Pillar 4 Robustness Summary (corruption: paraphrase, source: caller): original=0.0747, variant=0.0747, delta=0.0000, relative_drop=0.0000, robustness_ratio=1.0000
06:59:45 •     paraphrase: {'original_score': 0.07470703125, 'variant_score': 0.07470703125, 'delta': 0.0, 'relative_drop': 0.0, 'robustness_ratio': 1.0, 'corruption_variant': 'paraphrase', 'corruption_source': 'caller'}
06:59:45 • Pillar 4: Running robustness evaluation (corruption: entity_swap)...
06:59:45 • No corruption_dataloader supplied for variant 'entity_swap'; generating one from the original dataloader's clean prompts.


06:59:45 • Pillar 4: SKIPPING variant 'entity_swap' — entity_swap corruption unavailable: 'entity_swap' corruption strategy failed on all 256 examples (last error: EntitySwapCorruption requires a spaCy model. spaCy is installed but the 'en_core_web_sm' model is missing. Download it with:
    python -m spacy download en_core_web_sm
or pass a loaded spaCy model via the `nlp` argument / set_nlp().).. No robustness score is produced for this variant (it was NOT evaluated against the uncorrupted data).
06:59:45 •     entity_swap: {'corruption_variant': 'entity_swap', 'status': 'skipped', 'reason': "entity_swap corruption unavailable: 'entity_swap' corruption strategy failed on all 256 examples (last error: EntitySwapCorruption requires a spaCy model. spaCy is installed but the 'en_core_web_sm' model is missing. Download it with:\n    python -m spacy download en_core_web_sm\nor pass a loaded spaCy model via the `nlp` argument / set_nlp().)."}
06:59:45 • Pillar 4: Running robustness evaluatio

In [8]:
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## Section D: Pillar 3 — Stability

Run discovery multiple times with different random seeds and measure circuit overlap (Jaccard/Dice similarity). A stable circuit should be reproducible.

> This is expensive — each stability run re-runs discovery from scratch.

In [9]:
pipe.evaluate(
    pillars=["stability"],
    n_examples=64,  # lower than 256 -- each stability run re-runs discovery
    n_stability_runs=3,  # keep low for demo; use 5-10 for research
)

if pipe._eval_report:
    print(json.dumps(pipe._eval_report, indent=2, default=str))

06:59:47 • ==================================================
06:59:47 • Starting: Circuit Evaluation
06:59:47 • ==================================================
06:59:47 • [1/4] Loading config and model
06:59:47 ⚙ Configuration validated successfully.
Loaded pretrained model meta-llama/Llama-3.2-1B into HookedTransformer
06:59:52 🔧 Model loading: 4.81s
06:59:52 • Generating new IOI data for Llama-3.2-1B (samples=64, seed=42)
06:59:52 • Saved generated data to cache: cache/ioi/ioi_Llama-3.2-1B_64_seed42.csv
06:59:52 • [2/4] Reconstructing circuit
06:59:57 • [3/4] Running faithfulness evaluation
06:59:57 • Starting full faithfulness evaluation with pillars: ['stability']
06:59:57 🔧 Model: Llama-3.2-1B
06:59:57 • Task: ioi
06:59:57 • Algorithm: eap-ig
06:59:57 • Computing Pillar 3: Stability (3 runs)...
06:59:57 • Pillar 3: Running stability evaluation with 3 runs (seeds 42-44)...
06:59:57 • Baseline circuit has 370 nodes ({'InputNode': 1, 'AttentionNode': 354, 'MLPNode': 15})
06:59:57

100%|██████████| 128/128 [01:07<00:00,  1.90it/s]

07:01:08 •   Run 1: 528 scored nodes


07:01:09 • Stability re-discovery run 2/3 (seed=43, algo=eap-ig)
07:01:13 • Generating new IOI data for Llama-3.2-1B (samples=256, seed=43)
07:01:14 • Saved generated data to cache: cache/ioi/ioi_Llama-3.2-1B_256_seed43.csv


100%|██████████| 128/128 [01:07<00:00,  1.91it/s]

07:02:21 •   Run 2: 528 scored nodes


07:02:22 • Stability re-discovery run 3/3 (seed=44, algo=eap-ig)
07:02:26 • Generating new IOI data for Llama-3.2-1B (samples=256, seed=44)
07:02:27 • Saved generated data to cache: cache/ioi/ioi_Llama-3.2-1B_256_seed44.csv


100%|██████████| 128/128 [01:07<00:00,  1.90it/s]

07:03:35 •   Run 3: 528 scored nodes


07:03:36 • Pillar 3 Stability Metrics:
07:03:36 •   Mean Jaccard:     0.8287 ± 0.0119
07:03:36 •   Mean Dice:        0.9063 ± 0.0071
07:03:36 •   Mean Spearman ρ:  0.9211 ± 0.0063
07:03:36 •   Stable nodes:     316
07:03:36 •   Layer breakdown:  {0: 0.5851851851851851, 1: 0.6885964912280702, 2: 0.7982456140350878, 3: 0.8403418803418804, 4: 0.7555276848755109, 5: 0.7692307692307693, 6: 0.8913978494623657, 7: 0.8620689655172414, 8: 0.8768939393939394, 9: 0.8182539682539683, 10: 0.9059934318555009, 11: 0.7387642157757099, 12: 0.8540564373897707, 13: 0.8393939393939394, 14: 0.9025573192239859, 15: 0.9038800705467372}
07:03:36 ✓   Stability evaluation complete (219.4s)
07:03:36 •     Mean Jaccard: 0.8287
07:03:36 • ======================================================================
07:03:36 ✓ FAITHFULNESS EVALUATION COMPLETE
07:03:36 • ======================================================================
07:03:36 • Total time: 219.8s
07:03:36 • Pillar timings:
07:03:36 •   stability    

In [10]:
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## Section E: Pillar 6 — Generalization

Does the circuit transfer to a related but different task? For IOI, the standard generalization target is `double_io` (a variant with two indirect objects).

In [11]:
pipe.evaluate(
    pillars=["generalization"],
    n_examples=64,
    target_task="double_io",
)

if pipe._eval_report:
    print(json.dumps(pipe._eval_report, indent=2, default=str))

07:03:37 • ==================================================
07:03:37 • Starting: Circuit Evaluation
07:03:37 • ==================================================
07:03:37 • [1/4] Loading config and model
07:03:37 ⚙ Configuration validated successfully.
Loaded pretrained model meta-llama/Llama-3.2-1B into HookedTransformer
07:03:42 🔧 Model loading: 4.78s
07:03:42 • Loading cached IOI data for Llama-3.2-1B from cache/ioi/ioi_Llama-3.2-1B_64_seed42.csv
07:03:42 • Generating DoubleIO data for Llama-3.2-1B (N=64, seed=42, type=ABBA)
07:03:42 • Cached DoubleIO data → cache/double_io/double_io_Llama-3.2-1B_64_seed42.csv
07:03:42 • Target task for generalization: double_io
07:03:42 • [2/4] Reconstructing circuit
07:03:47 • [3/4] Running faithfulness evaluation
07:03:47 • Starting full faithfulness evaluation with pillars: ['generalization']
07:03:47 🔧 Model: Llama-3.2-1B
07:03:47 • Task: ioi
07:03:47 • Algorithm: eap-ig
07:03:47 • Computing Pillar 6: Generalization...
07:03:47 • Pillar 6: Ev

100%|██████████| 32/32 [00:01<00:00, 20.05it/s]

07:03:48 • Pillar 6 Source Score (ioi): 0.0332
07:03:48 • Evaluating circuit on target task: double_io...



100%|██████████| 32/32 [00:02<00:00, 15.63it/s]

07:03:50 • Pillar 6 Target Score (double_io): -0.5117
07:03:50 • Pillar 6 (ioi -> double_io): transfer_ratio undefined (source=0.0332, target=-0.5117); reporting status='invalid'.
07:03:50 ✓   Generalization evaluation complete (3.7s)
07:03:50 • ======================================================================
07:03:50 ✓ FAITHFULNESS EVALUATION COMPLETE
07:03:50 • ======================================================================
07:03:50 • Total time: 3.7s
07:03:50 • Pillar timings:
07:03:50 •   generalization :     3.7s
07:03:50 • ======================================================================
07:03:50 ✓ Full faithfulness report complete
07:03:50 • ==================================================
07:03:50 ✓ Completed: Circuit Evaluation in 13.2s
07:03:50 • ==================================================
"============================================================\nFAITHFULNESS EVALUATION REPORT\n============================================================\n\nPIL

In [12]:
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## Section F: All 6 Pillars at Once

For convenience, you can run all pillars in a single call. The framework runs them in optimal cost order internally.

> **Note:** If you already ran Sections A–E above, this cell re-runs discovery and all evaluations on a fresh pipeline. You can skip it to save time — it's included to demonstrate the single-call API.

In [13]:
# Run a fresh discovery to get a clean evaluation
pipe_full = Pipeline(
    "meta-llama/Llama-3.2-1B",
    task="ioi",
    output_dir="./results/eval_full",
)
pipe_full.discover(algorithm="eap-ig", n_examples=256, batch_size=2, ig_steps=5)

# All pillars
pipe_full.evaluate(
    pillars="all",
    n_examples=64,
    n_stability_runs=3,
    target_task="double_io",
)

if pipe_full._eval_report:
    print(json.dumps(pipe_full._eval_report, indent=2, default=str))

Loaded pretrained model meta-llama/Llama-3.2-1B into HookedTransformer
07:03:57 • ==================================================
07:03:57 • Starting: Circuit Discovery
07:03:57 • ==================================================
07:03:57 • [1/4] Loading and validating configuration
07:03:57 ⚙ Configuration validated successfully.
07:03:57 ⚙ Config: EAP-IG on ioi task, node level, 30% sparsity
07:03:57 • [2/4] Setting up model (model=meta-llama/Llama-3.2-1B)
Loaded pretrained model meta-llama/Llama-3.2-1B into HookedTransformer
07:04:01 🔧 Model loading: 4.81s
07:04:01 🔧 Model: meta-llama/Llama-3.2-1B (1.5B params, cuda device)
07:04:01 • [3/4] Running discovery algorithm (algorithm=eap-ig)
07:04:01 • Starting EAP-IG discovery algorithm
07:04:01 • Loading cached IOI data for Llama-3.2-1B from cache/ioi/ioi_Llama-3.2-1B_256_seed42.csv


100%|██████████| 128/128 [01:07<00:00,  1.90it/s]

07:05:13 • Converted 528 node scores from EAP-IG graph to CircuitKIT format.
07:05:13 • Saved unified CircuitScores → results/eval_full/eap-ig_meta-llama_Llama-3.2-1B_ioi_node_scores.json
07:05:13 • Saved 'scores' → results/eval_full/eap-ig_meta-llama_Llama-3.2-1B_ioi_node_scores.pt
07:05:13 • [4/4] Identifying nodes to prune
07:05:13 • 
--- Running Categorized Node Pruning (Scope: BOTH) ---
07:05:13 • Found 512 Attention Heads and 16 MLP layers to consider for pruning.
07:05:13 • Pruning 4 out of 16 MLP layers (30%).
07:05:13 • Targeting to prune 153 out of 512 total Attention Heads (30%).
07:05:13 • Actually pruned 153 Attention Heads.
07:05:13 •   - Layer 0: Pruned 16/32 heads.
07:05:13 •   - Layer 1: Pruned 16/32 heads.
07:05:13 •   - Layer 2: Pruned 16/32 heads.
07:05:13 •   - Layer 3: Pruned 8/32 heads.
07:05:13 •   - Layer 4: Pruned 13/32 heads.
07:05:13 •   - Layer 5: Pruned 10/32 heads.
07:05:13 •   - Layer 6: Pruned 5/32 heads.
07:05:13 •   - Layer 7: Pruned 8/32 heads.
07:05

Loaded pretrained model meta-llama/Llama-3.2-1B into HookedTransformer
07:05:18 🔧 Model loading: 4.67s
07:05:18 • Loading cached IOI data for Llama-3.2-1B from cache/ioi/ioi_Llama-3.2-1B_64_seed42.csv
07:05:18 • Loading cached IOI data for Llama-3.2-1B from cache/ioi/ioi_Llama-3.2-1B_64_seed42.csv
07:05:18 • Loading cached DoubleIO data from cache/double_io/double_io_Llama-3.2-1B_64_seed42.csv
07:05:18 • Target task for generalization: double_io
07:05:18 • [2/4] Reconstructing circuit
07:05:23 • [3/4] Running faithfulness evaluation
07:05:23 • Starting full faithfulness evaluation with pillars: ['patching', 'ablation', 'baselines', 'robustness', 'stability', 'generalization', 'intervention_reliability']
07:05:23 🔧 Model: Llama-3.2-1B
07:05:23 • Task: ioi
07:05:23 • Algorithm: eap-ig
07:05:23 • Computing Pillar 1: Causal Patching...
07:05:23 • Pillar 1: Running causal patching evaluation...


100%|██████████| 32/32 [00:02<00:00, 12.87it/s]

...

100%|██████████| 128/128 [01:06<00:00,  1.92it/s]

07:12:52 • Converted 528 node scores from EAP-IG graph to CircuitKIT format.
07:12:52 • Loading cached IOI data for Llama-3.2-1B from cache/ioi/ioi_Llama-3.2-1B_256_seed76.csv



100%|██████████| 128/128 [00:10<00:00, 12.78it/s]

07:13:09 • Pillar 7 done: R1=0.464 R2=0.504 R3=0.000 reliability=0.000
07:13:09 ✓   Intervention reliability complete (181.5s) — index=0.0000


07:13:11 • ======================================================================
07:13:11 ✓ FAITHFULNESS EVALUATION COMPLETE
07:13:11 • ======================================================================
07:13:11 • Total time: 467.6s
07:13:11 • Pillar timings:
07:13:11 •   patching       :     6.1s
07:13:11 •   ablation       :     5.2s
07:13:11 •   baselines      :    43.9s
07:13:11 •   robustness     :     3.3s
07:13:11 •   stability      :   219.3s
07:13:11 •   generalization :     3.7s
07:13:11 •   intervention_reliability:   181.5s
07:13:11 • ======================================================================
07:13:11 ✓ Full faithfulness report complete
07:13:11 • ==================================================
07:13:11 ✓ Completed: Circuit Evaluation in 477.3s
07:13:11 • ==================================================
"============================================================\nFAITHFULNESS EVALUATION REPORT\n========================================================

## How to Read Faithfulness Numbers

A circuit is "good enough" when:

- **Patching score** is high (ablating circuit → large performance drop)
- **Ablation score** is high (circuit alone retains most behavior)
- **Stability** Jaccard > 0.5 across seeds
- **Baselines** — circuit outperforms random selection at the same sparsity
- **Robustness** — scores are stable across corruption types
- **Generalization** — some transfer to related tasks (depends on task similarity)

No single pillar is definitive. The framework is designed to give a **multi-dimensional** view of circuit quality.